# Ordinary Least Squares

**DS4DH · Module 05 — Regression Analysis**

*Technique:* OLS, and the fact that a single-dummy regression is a group-mean difference

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Sagaustus/ds4dh-colab-pack/blob/main/notebooks/05a_ols.ipynb)

Data: `merged_dataset.csv` — from the `data/` folder of this pack.

---

In [ ]:
# Setup — run this first.
import os, warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import statsmodels.api as sm

# This notebook reads the CSVs sitting next to it. In Colab, upload them from
# the pack's data/ folder when prompted. The exists() guard means a re-run
# part-way through a session will not ask you to upload all over again.
NEEDED = ['merged_dataset.csv']
missing = [f for f in NEEDED if not os.path.exists(f)]
if missing:
    try:
        from google.colab import files
        print('Upload from the data/ folder of the pack: ' + ', '.join(missing))
        files.upload()
    except ImportError:
        raise SystemExit('Place these next to the notebook: ' + ', '.join(missing))

df       = pd.read_csv('merged_dataset.csv')
CITIES = ['Montréal', 'Toronto', 'Edmonton', 'Vancouver']

print(f'Loaded. df has {len(df):,} rows and {df.shape[1]} columns.')

## What this notebook does

Regression is usually introduced as a new and more powerful thing. It is more
general, but the first model here computes something you already know.

Regressing renter STIR on a single 0/1 immigrant indicator gives a coefficient
that **is** the difference in group means. Seeing that identity is the fastest way
to understand what a coefficient means before the models get complicated.

In [ ]:
csd = df.dropna(subset=['csd_code'])

reg_df = csd[csd['immigrant_status'].isin(['Immigrant', 'Non-immigrants'])
             & csd['cma'].isin(CITIES)].dropna(subset=['Renter']).copy()
reg_df['is_immigrant'] = (reg_df['immigrant_status'] == 'Immigrant').astype(int)

print(f'{len(reg_df)} rows — one per (CSD, immigrant status) with a renter STIR')
print(reg_df['immigrant_status'].value_counts().to_string())

In [ ]:
# Model A: one binary predictor.
y = reg_df['Renter']
X = sm.add_constant(reg_df[['is_immigrant']])
model_a = sm.OLS(y, X).fit()

print(model_a.summary().tables[1])

## Reading the coefficient table

- **`const`** — the predicted value when every predictor is 0. Here: the mean
  renter STIR for non-immigrant households.
- **`is_immigrant`** — how much the prediction changes when the indicator goes
  from 0 to 1. Here: the immigrant/non-immigrant difference in percentage points.
- **`P>|t|`** — the same p-value logic as Module 04, now for a coefficient.
- **`[0.025 0.975]`** — the 95% confidence interval. If it straddles zero, the
  coefficient is not distinguishable from no effect.

In [ ]:
# The identity, checked directly.
mean_imm = reg_df[reg_df['is_immigrant'] == 1]['Renter'].mean()
mean_non = reg_df[reg_df['is_immigrant'] == 0]['Renter'].mean()

print(f'mean renter STIR, non-immigrant : {mean_non:.4f}')
print(f'mean renter STIR, immigrant     : {mean_imm:.4f}')
print(f'difference                      : {mean_imm - mean_non:+.4f}')
print()
print(f'OLS const                       : {model_a.params["const"]:.4f}')
print(f'OLS is_immigrant coefficient    : {model_a.params["is_immigrant"]:+.4f}')
print()
print('Identical. A regression on one dummy IS a difference of means, computed')
print('by a method general enough to take more predictors.')

### 🔧 Your turn 1

Flip the coding: define `is_non_immigrant = 1 - is_immigrant`, refit, and compare.

What happens to `const`? What happens to the coefficient? Which of the two
numbers carries the comparison, and which is just a reference point?

## What the model explains

R² is the share of variance in the outcome the model accounts for. A near-zero R²
with a large, well-estimated coefficient is entirely possible and often correct —
it means the effect is real but small relative to everything else that varies.

In [ ]:
print(f'R-squared      {model_a.rsquared:.4f}')
print(f'observations   {int(model_a.nobs)}')
print(f'coefficient    {model_a.params["is_immigrant"]:+.3f} pp')
print(f'p-value        {model_a.pvalues["is_immigrant"]:.4f}')
print()
print(f'Immigrant status explains {model_a.rsquared:.1%} of the variation in')
print('renter STIR across these rows. Almost everything is something else —')
print('and "which city" is the obvious first candidate.')

In [ ]:
# Continuous predictors work the same way, with a different unit.
sub = reg_df.dropna(subset=['rent_income']).copy()
sub['income_10k'] = sub['rent_income'] / 10000

m = sm.OLS(sub['Renter'], sm.add_constant(sub[['income_10k']])).fit()
print(m.summary().tables[1])
print()
print(f'Each extra $10,000 of renter household income is associated with a')
print(f'{m.params["income_10k"]:+.3f} pp change in renter STIR.')

### 🔧 Your turn 2

Change the scaling from `/ 10000` to `/ 1000` and refit.

The coefficient changes by a factor of ten. Does the p-value change? Does R²?
What does that tell you about which parts of a regression output depend on your
units and which do not?

<details markdown="1">
<summary><b>What you should have seen</b> — click to expand</summary>

**Your turn 1.** `const` becomes the immigrant mean and the coefficient flips
sign, with the same magnitude and the same p-value. The intercept is only a
reference point — it depends entirely on which category you coded as zero. The
coefficient carries the comparison. This is why "the constant was significant" is
almost never an interesting sentence.

**Your turn 2.** The coefficient scales by ten; the p-value and R² do not move at
all. Units affect the *size* of a coefficient and nothing about the evidence for
it. A coefficient is meaningless without its units, and a coefficient chosen to
look large by rescaling is a presentational trick, not a finding.

</details>

## Where this stops

Model A says immigrant renters spend about 0.38pp more, and cannot distinguish
that from zero. It also cannot distinguish it from a "lives in an expensive city"
effect. Notebook 05b separates the two.